In [2]:
import torch as t
import torch.nn as nn

In [3]:
pairwise = t.arange(0, 256, dtype=t.float32)
pairwise = pairwise.reshape(-1, 2)

In [4]:
pairwise.size()

torch.Size([128, 2])

In [5]:
theta = 10000 ** (-2 * t.arange(0, 256//2) / 256)

theta = theta.unsqueeze(-1)

sq_len = t.arange(1024, dtype=t.float32).unsqueeze(-1)

In [6]:
theta.T.shape

torch.Size([1, 128])

In [7]:
ooo = sq_len @ theta.T

In [8]:
ooo.shape

torch.Size([1024, 128])

In [9]:
cp = t.view_as_complex(pairwise)

In [10]:
theta = 10000 ** (-2 * t.arange(0, 128)/256)

In [11]:
ee = t.exp(1j * t.arange(128) * theta)

In [12]:
rot = cp * ee
realrot = t.view_as_real(rot)

In [13]:
realrot

tensor([[ 0.0000e+00,  1.0000e+00],
        [-1.2111e+00,  3.3960e+00],
        [-5.5770e+00,  3.1460e+00],
        [-9.1318e+00, -1.2692e+00],
        [-9.1933e+00, -7.7771e+00],
        [-5.6554e+00, -1.3748e+01],
        [ 1.6414e-01, -1.7691e+01],
        [ 6.7952e+00, -1.9360e+01],
        [ 1.3221e+01, -1.9241e+01],
        [ 1.8951e+01, -1.8051e+01],
        [ 2.3874e+01, -1.6464e+01],
        [ 2.8070e+01, -1.5002e+01],
        [ 3.1685e+01, -1.4037e+01],
        [ 3.4844e+01, -1.3815e+01],
        [ 3.7615e+01, -1.4495e+01],
        [ 3.9996e+01, -1.6165e+01],
        [ 4.1920e+01, -1.8859e+01],
        [ 4.3266e+01, -2.2561e+01],
        [ 4.3877e+01, -2.7200e+01],
        [ 4.3577e+01, -3.2651e+01],
        [ 4.2195e+01, -3.8737e+01],
        [ 3.9585e+01, -4.5233e+01],
        [ 3.5640e+01, -5.1873e+01],
        [ 3.0308e+01, -5.8365e+01],
        [ 2.3597e+01, -6.4406e+01],
        [ 1.5583e+01, -6.9701e+01],
        [ 6.4029e+00, -7.3973e+01],
        [-3.7509e+00, -7.698

In [14]:
class RoPE(nn.Module):
    def __init__(self, dim: int, seq_len: int):
        super().__init__()

        # theta = 10000^{-2i/d} part
        self.theta = 10000 ** (-2 * t.arange(0, dim//2, dtype=t.float32) / dim).unsqueeze(-1) # shape = [128, 1]

        #converting to e^{i m theta}
        self.e = t.exp(1j * (t.arange(seq_len, dtype=t.float32).unsqueeze(-1) @ self.theta.T))  # shape = [1024, 1] x [1, 128] = [1024, 128]. 
        
        self.e = self.e.unsqueeze(0) # [1024, 128] -> [1, 1024, 128]

        self.register_buffer("eee", self.e)

    def forward(self, xq, xk):

        # pairing them up along the embedding dimension: [1, 2, 3, 4] -> [[1, 2], [3, 4]]
        xq_paired = xq.reshape(*xq.shape[:-1], -1, 2)
        xk_paired = xk.reshape(*xk.shape[:-1], -1, 2)

        # clipping length of buffer in case of shorter sequence.
        e = self.e[:, :xq_comp.shape[1], :]

        # view them as complex: [[1, 2], [3, 4]] -> [[1, 2j], [3, 4j]]
        xq_comp = t.view_as_complex(xq_paired)
        xk_comp = t.view_as_complex(xk_paired)

        # rotato.
        xq_rot = xq_comp * self.e
        xk_rot = xk_comp * self.e

        # back to real
        xq_out = t.view_as_real(xq_rot).flatten(-2)
        xk_out = t.view_as_real(xk_rot).flatten(-2)

        return xq_out, xk_out

        

In [15]:
positions = t.arange(6)
# Broadcasting subtraction creates a matrix where cell (i, j) is i - j
distances = positions.unsqueeze(1) - positions.unsqueeze(0)


In [16]:
test_input = t.randn(8, 1024, 256)

In [17]:
rope = RoPE(256, 1024)
output = rope(test_input, test_input)

UnboundLocalError: cannot access local variable 'xq_comp' where it is not associated with a value

In [ ]:
test_input

tensor([[[ 1.0245, -0.1864,  0.6391,  ..., -0.9386, -0.2570,  0.6142],
         [ 0.3479, -0.8940,  0.1326,  ..., -0.4363,  0.2171,  0.9808],
         [-0.6447,  0.4136, -2.0740,  ...,  1.2737,  0.8169,  0.0516],
         ...,
         [-0.5966, -0.2607,  0.5571,  ..., -0.7402,  0.1207, -0.5629],
         [-1.2659,  0.9037, -0.2868,  ..., -1.8195,  1.3039,  0.4154],
         [-1.5451,  2.0778,  1.2370,  ..., -1.4833, -1.9087, -0.8551]],

        [[ 0.6149,  0.8043,  0.7696,  ...,  1.3171,  0.3907,  0.8163],
         [-0.8231, -1.0517, -0.6605,  ...,  0.3925, -0.1846,  1.2665],
         [ 0.8044,  0.6206, -0.0221,  ...,  2.5207,  1.8671, -0.3098],
         ...,
         [ 0.5478, -0.8115,  1.1881,  ..., -0.8844,  0.2915, -1.4898],
         [ 0.0602, -0.5942, -0.6637,  ...,  0.0683, -1.9156, -0.8463],
         [ 1.8794, -0.4010, -0.9524,  ...,  0.6451, -1.0768, -0.5036]],

        [[ 0.9445,  0.6428,  0.1193,  ..., -0.9795, -0.0070,  0.3021],
         [-1.1725,  0.2840,  0.1427,  ...,  0

In [ ]:
output

(tensor([[[ 1.0245, -0.1864,  0.6391,  ..., -0.9386, -0.2570,  0.6142],
          [ 0.9403, -0.1903, -0.5646,  ..., -0.4362,  0.2170,  0.9808],
          [-0.1078, -0.7583, -0.2709,  ...,  1.2737,  0.8169,  0.0517],
          ...,
          [ 0.6011,  0.2502, -0.5487,  ..., -0.6059,  0.1816, -0.5463],
          [ 1.4544,  0.5514, -0.0259,  ..., -1.7714,  1.2505,  0.5558],
          [ 1.2861,  2.2473, -1.1614,  ..., -1.4525, -1.8034, -1.0593]],
 
         [[ 0.6149,  0.8043,  0.7696,  ...,  1.3171,  0.3907,  0.8163],
          [ 0.4403, -1.2609, -0.1470,  ...,  0.3927, -0.1847,  1.2665],
          [-0.8991,  0.4732, -0.1447,  ...,  2.5207,  1.8672, -0.3094],
          ...,
          [-0.5334,  0.8210,  0.4797,  ..., -0.9230,  0.4529, -1.4489],
          [-0.5277,  0.2797, -1.1254,  ...,  0.0632, -1.8113, -1.0512],
          [ 0.3844, -1.8829,  0.9269,  ...,  0.5258, -1.0151, -0.6187]],
 
         [[ 0.9445,  0.6428,  0.1193,  ..., -0.9795, -0.0070,  0.3021],
          [-0.8724, -0.8332,

In [ ]:
W = 3

allowed = (distances >= 0) & (distances < W)

attn_mask = t.where(allowed, 0.0, float('-inf'))

In [ ]:
attn_mask

tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [-inf, 0., 0., 0., -inf, -inf],
        [-inf, -inf, 0., 0., 0., -inf],
        [-inf, -inf, -inf, 0., 0., 0.]])

In [21]:
class ALiBi(nn.Module):
    def __init__(self, heads: int, seq_len: int, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.seq_len = seq_len

        positions = t.arange(seq_len, dtype=t.float32)
        distances = positions.unsqueeze(1) - positions.unsqueeze(0)

        # negative so that the penalties remain negative.
        penals = -t.abs(distances)

        slopes = 2 ** -(t.arange(1, heads+1, dtype=t.float32) * 8/heads)
        slopes = slopes.reshape(heads, 1, 1)

        self.register_buffer("alibi_bias", slopes * penals)

    def forward(self):
        causal_mask = t.triu(t.full((self.seq_len, self.seq_len), float('-inf')), diagonal=1)
        final_attn_mask = self.alibi_bias + causal_mask 

        return final_attn_mask

In [ ]:
heads = t.arange(8)
slopes = 2 ** -(t.arange(8, dtype=t.float32))
slopes = slopes.view(len(heads), 1, 1)
penal = -t.abs(distances)

In [ ]:
(slopes * penal)

tensor([[[ 0.0000, -1.0000, -2.0000, -3.0000, -4.0000, -5.0000],
         [-1.0000,  0.0000, -1.0000, -2.0000, -3.0000, -4.0000],
         [-2.0000, -1.0000,  0.0000, -1.0000, -2.0000, -3.0000],
         [-3.0000, -2.0000, -1.0000,  0.0000, -1.0000, -2.0000],
         [-4.0000, -3.0000, -2.0000, -1.0000,  0.0000, -1.0000],
         [-5.0000, -4.0000, -3.0000, -2.0000, -1.0000,  0.0000]],

        [[ 0.0000, -0.5000, -1.0000, -1.5000, -2.0000, -2.5000],
         [-0.5000,  0.0000, -0.5000, -1.0000, -1.5000, -2.0000],
         [-1.0000, -0.5000,  0.0000, -0.5000, -1.0000, -1.5000],
         [-1.5000, -1.0000, -0.5000,  0.0000, -0.5000, -1.0000],
         [-2.0000, -1.5000, -1.0000, -0.5000,  0.0000, -0.5000],
         [-2.5000, -2.0000, -1.5000, -1.0000, -0.5000,  0.0000]],

        [[ 0.0000, -0.2500, -0.5000, -0.7500, -1.0000, -1.2500],
         [-0.2500,  0.0000, -0.2500, -0.5000, -0.7500, -1.0000],
         [-0.5000, -0.2500,  0.0000, -0.2500, -0.5000, -0.7500],
         [-0.7500, -0

In [ ]:
distances.shape

torch.Size([6, 6])

In [ ]:
slopes

tensor([0.5000, 0.2500, 0.1250, 0.0625, 0.0312, 0.0156, 0.0078, 0.0039])

In [22]:
alibi = ALiBi(8, 1024)
test_input = t.randn(4, 8, 1024, 256)

output = alibi()

In [23]:
output

tensor([[[ 0.0000e+00,        -inf,        -inf,  ...,        -inf,
                 -inf,        -inf],
         [-5.0000e-01,  0.0000e+00,        -inf,  ...,        -inf,
                 -inf,        -inf],
         [-1.0000e+00, -5.0000e-01,  0.0000e+00,  ...,        -inf,
                 -inf,        -inf],
         ...,
         [-5.1050e+02, -5.1000e+02, -5.0950e+02,  ...,  0.0000e+00,
                 -inf,        -inf],
         [-5.1100e+02, -5.1050e+02, -5.1000e+02,  ..., -5.0000e-01,
           0.0000e+00,        -inf],
         [-5.1150e+02, -5.1100e+02, -5.1050e+02,  ..., -1.0000e+00,
          -5.0000e-01,  0.0000e+00]],

        [[ 0.0000e+00,        -inf,        -inf,  ...,        -inf,
                 -inf,        -inf],
         [-2.5000e-01,  0.0000e+00,        -inf,  ...,        -inf,
                 -inf,        -inf],
         [-5.0000e-01, -2.5000e-01,  0.0000e+00,  ...,        -inf,
                 -inf,        -inf],
         ...,
         [-2.5525e+02, -2